In [ ]:
# Saving figures for the book (website and print).
# save_for_print(fig, name, width_in) makes sure that no text is smaller than
# MIN_PT points once the figure is printed width_in inches wide (the book's text
# is 4.5 in wide, so a figure at 90% is 4.05 in), hides the tick numbers of 3D
# axes, and saves img/<name>.png at PRINT_DPI for that printed width.
import os
import matplotlib.text
from mpl_toolkits.mplot3d import Axes3D

IMG_DIR = 'img'      # where the images go (the book's img/ folder)
MIN_PT = 7           # smallest text size in print, in points
PRINT_DPI = 400      # resolution at the printed size

def save_for_print(fig, name, width_in, min_pt=MIN_PT, hide_3d_ticks=True):
    if hide_3d_ticks:
        for ax in fig.axes:
            if isinstance(ax, Axes3D):
                ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    for _ in range(3):  # enlarging text changes the figure's extent, so repeat
        fig.canvas.draw()
        scale = width_in / fig.get_tightbbox().width   # printed size / drawn size
        texts = [t for t in fig.findobj(matplotlib.text.Text)
                 if t.get_visible() and t.get_text().strip()]
        small = [t for t in texts if t.get_fontsize() * scale < min_pt - 0.05]
        for t in small:
            t.set_fontsize(min_pt / scale)
        for ax in fig.axes:  # tick labels are re-created when drawn, so set them on the axes too
            for axis in ('x', 'y'):
                labels = getattr(ax, f'get_{axis}ticklabels')()
                if labels and labels[0].get_fontsize() * scale < min_pt - 0.05:
                    ax.tick_params(axis=axis, labelsize=min_pt / scale)
        if not small:
            break
    fig.canvas.draw()
    dpi = PRINT_DPI * width_in / fig.get_tightbbox().width
    os.makedirs(IMG_DIR, exist_ok=True)
    fig.savefig(os.path.join(IMG_DIR, name + '.png'), dpi=dpi, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    print(f'{name}.png: {width_in} in wide in print, smallest text {min_pt} pt')


### Quantum Harmonic Oscillator: Eigenvalues and Eigenfunctions

This notebook calculates and visualizes the eigenvalues and eigenfunctions of the quantum mechanical one-dimensional harmonic oscillator. The approach involves discretizing the Hamiltonian operator into a matrix and then solving the resulting eigenvalue problem numerically.

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh

# Define physical constants (using 'natural units' for simplicity, or specific values for clarity)
# We'll use a scaled system where m=1, h_bar=1, omega=1 for simplicity
# You can change these to actual physical constants if needed
# For example: h_bar = 1.0545718e-34 # Reduced Planck's constant in J*s
# m = 9.1093837e-31 # Mass of electron in kg
# omega = 1e15 # Angular frequency in rad/s

m = 1.0  # Mass
h_bar = 1.0  # Reduced Planck's constant
omega = 1.0  # Angular frequency

#### 1. Define the Potential and Spatial Grid

We define the harmonic oscillator potential $V(x) = \frac{1}{2}m\omega^2x^2$. We also set up a spatial grid `x` over which we will solve the Schrödinger equation.

In [ ]:
# Spatial grid parameters
N = 500  # Number of spatial points
x_max = 5.0  # Maximum x value
x_min = -x_max # Minimum x value
x = np.linspace(x_min, x_max, N) # Spatial grid
dx = x[1] - x[0] # Spatial step size

# Harmonic oscillator potential
V = 0.5 * m * omega**2 * x**2

#### 2. Construct the Hamiltonian Matrix

The time-independent Schrödinger equation is $H\psi = E\psi$, where $H$ is the Hamiltonian operator.
For a 1D harmonic oscillator, $H = -\frac{\hbar^2}{2m}\frac{d^2}{dx^2} + V(x)$.

We approximate the second derivative using a finite difference method:
$\frac{d^2\psi}{dx^2} \approx \frac{\psi(x+\Delta x) - 2\psi(x) + \psi(x-\Delta x)}{(\Delta x)^2}$

This leads to a tridiagonal matrix representation of the kinetic energy term. The potential energy term is a diagonal matrix where each element is $V(x_i)$.

In [ ]:
# Kinetic energy term (second derivative approximated by finite differences)
# The term is -h_bar^2 / (2 * m * dx^2)
kinetic_coeff = -h_bar**2 / (2 * m * dx**2)

# Create the kinetic energy matrix (tridiagonal)
main_diag = np.full(N, -2 * kinetic_coeff) # -2 * (-h_bar^2 / (2mdx^2)) = h_bar^2 / (mdx^2)
off_diag = np.full(N - 1, kinetic_coeff) # -h_bar^2 / (2mdx^2)

KE_matrix = np.diag(main_diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)

# Potential energy matrix (diagonal matrix)
PE_matrix = np.diag(V)

# Total Hamiltonian matrix
H_matrix = KE_matrix + PE_matrix

#### 3. Solve for Eigenvalues and Eigenfunctions

We use `scipy.linalg.eigh` to find the eigenvalues (energies) and eigenvectors (wavefunctions) of the Hamiltonian matrix. `eigh` is used for Hermitian (symmetric in real case) matrices, which our Hamiltonian is.

In [ ]:
# Solve the eigenvalue problem
# energies will contain the eigenvalues (energies)
# wavefunctions will contain the eigenvectors (eigenfunctions)
energies, wavefunctions = eigh(H_matrix)

# Normalize wavefunctions (optional, but good practice)
# The wavefunctions are already normalized by eigh such that sum(|psi|^2 * dx) = 1
# for i in range(wavefunctions.shape[1]):
#     wavefunctions[:, i] /= np.sqrt(np.sum(wavefunctions[:, i]**2) * dx)

print("First 5 Eigenvalues (Energies):")
for i in range(5):
    print(f"E_{i} = {energies[i]:.4f}")

#### 4. Visualize the Results

We plot the potential energy curve, the calculated energy levels (eigenvalues), and the first few eigenfunctions (wavefunctions). Note that the eigenfunctions are sometimes positive and sometimes negative, but their square (probability density) is always positive.

In [ ]:
import matplotlib as mpl

# Set font family preferences to prioritize Times New Roman
mpl.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['mathtext.fontset'] = 'cm' # Use Computer Modern for math text, which is serif

plt.figure(figsize=(10, 8))

plt.rcParams.update({'font.size': 14}) # Global font size increase

scaling_factor = 4.0 # Scaling factor for wavefunction amplitude, reduced from 5.0

# Plot the potential
plt.plot(x, V, 'k', label='Potential $V(x)$') # Changed 'k--' to 'k' for a continuous black line

# Plot the first few energy levels (eigenvalues)
num_states_to_plot = 5
for i in range(num_states_to_plot):
    # Calculate shifted wavefunction values
    psi_values = wavefunctions[:, i]
    shifted_psi = psi_values * scaling_factor + energies[i]

    # Plot positive and negative parts with different colors
    # Create arrays with NaN for parts that should not be plotted
    psi_pos_plot = np.where(psi_values >= 0, shifted_psi, np.nan)
    psi_neg_plot = np.where(psi_values < 0, shifted_psi, np.nan)

    # Plot the positive part (red)
    plt.plot(x, psi_pos_plot, color='red', label="_nolegend_")
    # Plot the negative part (blue)
    plt.plot(x, psi_neg_plot, color='blue', label="_nolegend_")

    # Fill between the wavefunction and its energy level
    plt.fill_between(x, shifted_psi, energies[i], where=(psi_values >= 0), color='red', alpha=0.1, interpolate=True)
    plt.fill_between(x, shifted_psi, energies[i], where=(psi_values < 0), color='blue', alpha=0.1, interpolate=True)

    # Plot the probability density
    # Probability density is psi^2, scaled and shifted to the energy level
    probability_density = psi_values**2
    # A new scaling factor might be needed for probability density for good visibility
    scaling_factor_pd = 50.0 # Increased scaling factor for probability density
    shifted_pd = probability_density * scaling_factor_pd + energies[i]
    plt.plot(x, shifted_pd, color='cyan', linestyle='-', label="_nolegend_")

    # Plot the energy level line
    plt.axhline(y=energies[i], color='gray', linestyle=':', linewidth=0.7)

    # Find and plot nodes (zero crossings)
    zero_crossings_indices = np.where(np.diff(np.sign(psi_values)))[0]
    node_x_coords = []
    for j in zero_crossings_indices:
        x1, x2 = x[j], x[j+1]
        y1, y2 = psi_values[j], psi_values[j+1]
        if abs(y2 - y1) > 1e-9:
            x_node = x1 + (0 - y1) * (x2 - x1) / (y2 - y1)
            node_x_coords.append(x_node)

    # Plot nodes as purple dots
    if node_x_coords:
        plt.plot(node_x_coords, np.full_like(node_x_coords, energies[i]), 'o', color='purple', markersize=4, label="_nolegend_")

    # Add labels to the left for each eigenfunction
    y_offset = 0.05 # Initial offset
    if i == 4: # Apply additional offset only for the 5th state (psi_4)
        y_offset += 0.15 # Additional offset for psi_4 labels
    if i == 3: # Apply additional offset only for the 5th state (psi_4)
        y_offset += 0.1 # Additional offset for psi_4 labels
    if i == 2: # Apply additional offset only for the 5th state (psi_4)
        y_offset += 0.05 # Additional offset for psi_4 labels
    plt.text(x_min + 1.0, energies[i] + y_offset, rf'$\psi_{i}(x)$', verticalalignment='bottom', color='black', fontfamily='serif',bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="black", lw=0.8, alpha=0.6))
    # Add label for probability density
    plt.text(x_max - 1.1, energies[i] + y_offset, rf'$|\psi_{i}(x)|^2$', verticalalignment='bottom', color='black', fontfamily='serif',bbox=dict(boxstyle="round,pad=0.1", fc="cyan", ec="black", lw=0.8, alpha=0.6))

#plt.title('Quantum Harmonic Oscillator: Eigenvalues and Eigenfunctions')
plt.xlabel('Position $x$')
plt.ylabel('Energy / Wavefunction Amplitude')

# Set y-axis limits and ticks based on energy levels
plt.ylim(0, energies[num_states_to_plot-1] + 1.0) # Adjust y-limit dynamically, slightly above the last plotted energy
# Revert y-ticks to show numerical values and units
plt.yticks(energies[:num_states_to_plot], [r'{:.1f} $\hbar\omega$'.format(e) for e in energies[:num_states_to_plot]])

# Adjust x-axis limits to accommodate labels on both sides
plt.xlim(x_min, x_max) # Extend x-axis to accommodate labels on both sides
plt.xticks([x_min, 0, x_max], ['', '0', '']) # Set x-axis ticks to show only 0

# Remove legend
# plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7) # Only show y-axis grid lines
save_for_print(plt.gcf(), 'HarmOsc', 4.05)
plt.show()